<center>

# Rocking Map Globe


</center>

## Parallel-beam electron channeling simulation for GaAs

### Notebook overview

This notebook calculates the orientation-dependent electron channeling in GaAs using a multibeam Bloch-wave approach. For each incident beam direction, a set of relevant diffraction beams is selected, the corresponding Bloch-wave eigenvalue problem is solved, and the electron intensity at the Ga and As atomic sites is averaged over the specimen thickness.

The final quantity used in the rocking maps is the channeling factor

$$
f_{\mathrm{channeling}}
=
\frac{J_{\mathrm{As}}}{J_{\mathrm{Ga}}},
$$

where $J_{\mathrm{As}}$ and $J_{\mathrm{Ga}}$ are the thickness-averaged electron intensities at the As and Ga sites.

The long-term aim is to include this orientation-dependent intensity redistribution in quantitative EDS. The conventional Cliff-Lorimer relation is

$$
\frac{C_{\mathrm{As}}}{C_{\mathrm{Ga}}}
=
k_{\mathrm{As,Ga}}
\frac{I_{\mathrm{As}}}{I_{\mathrm{Ga}}}.
$$

If the measured X-ray intensities are additionally affected by the different electron intensities at the two atomic sites, a channeling correction can be introduced as

$$
\frac{C_{\mathrm{As}}}{C_{\mathrm{Ga}}}
=
k_{\mathrm{As,Ga}}
\frac{I_{\mathrm{As}}}{I_{\mathrm{Ga}}}
\frac{J_{\mathrm{Ga}}}{J_{\mathrm{As}}}.
$$

Using the definition of $f_{\mathrm{channeling}}$ above,

$$
\frac{C_{\mathrm{As}}}{C_{\mathrm{Ga}}}
=\frac{k_{\mathrm{As,Ga}}}{f_{\mathrm{channeling}}}\frac{I_{\mathrm{As}}}{I_{\mathrm{Ga}}}
$$

The present notebook considers a **parallel incident electron beam**. It therefore calculates the channeling response for individual beam orientations and generates rocking maps around selected zone axes.

For a convergent beam, the calculated orientation-dependent results are stored as an `.npz` file. The second notebook loads this map and combines the contributions from the different incident directions within the convergence angle.

### Computational note

The calculation is computationally expensive because the diffraction beams are selected and the Bloch-wave eigenvalue problem is solved again for every point of the rocking map. A full map can therefore require several hours. Increasing the number of CPU cores does not currently lead to a proportional reduction in calculation time, so further optimization of the beam selection and repeated matrix calculations would be required for substantially faster calculations.

In [1]:
import os

# Avoid nested BLAS threading when the notebook itself is parallelized externally.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import time
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from tqdm import tqdm

In [2]:
available_cores = len(os.sched_getaffinity(0))

print("CPU cores available to notebook:", available_cores)

CPU cores available to notebook: 10


## Variables



In [ ]:
# Orientation, angles inspired by Recipro
u,v,w = 1,1,0
zone = np.array([u, v, w], dtype=float)
n_surface = normal_hkl = n_zone = zone / np.linalg.norm(zone) # n_surface and normal_hkl are not automatically the same, but to make things easier, I treated them as the same. This is only used for the electron density distribution map

special_name_for_file = ""

# Scan range means the displacement of the middle. 
scan_range1=50 # Up to 90 degrees (dtheta)
scan_range2=50 # Up to 180 degrees (dpsi)
stepsize=0.1   # Be aware of the processing time

N_BEAMS=169
HKL_MAX=10

# # 50 kV Acceleration Voltage, Tons request
# K = 18.67          # electron wavevector [Å^-1]

# 200 kV Acceleration Voltage
K = 39.84063745           # electron wavevector [Å⁻¹]

# # 1000 kV Acceleration Voltage, Tons request
# K = 114.69        # electron wavevector [Å^-1]


# GaAs Specimin
a = 5.653                 # GaAs lattice parameter [Å]
thickness_A = 630         # [Å] its a global thicknes it does not change with a new rotation

## Structure factor and Fourier coefficient of the crystal potential

For a cubic lattice, the magnitude of the scattering vector is

$$
s = \frac{\sqrt{h^2+k^2+l^2}}{2a},
$$

where $a$ is the lattice constant. The corresponding atomic scattering factors
$f_{\mathrm{Ga}}(s)$ and $f_{\mathrm{As}}(s)$ are taken from the scattering-factor table (International Tables for Crystallography, Volume C (2006), p. 554 The nearest tabulated scattering-vector value is used).

For the FCC lattice, reflections are allowed only if $h,k,l$ are either all even or all odd.

For zinc-blende GaAs, the structure factor is

$$
F_{hkl}
=
4\left[
f_{\mathrm{Ga}}
+
f_{\mathrm{As}}
\exp\left(
2\pi i\frac{h+k+l}{4}
\right)
\right].
$$

The Fourier coefficient of the crystal potential used in the Bloch-wave calculation is

$$
U_{\mathbf{g}}
=
\frac{F_{hkl}}{\pi \cdot V_{\mathrm{cell}}},
$$

with

$$
V_{\mathrm{cell}} = a^3.
$$

For forbidden FCC reflections,

$$
U_{\mathbf{g}} = 0.
$$

In [4]:
# Column 0: s [Å^-1]
# Column 1: Ga
# Column 2: As

scattering_table = np.array([
    [0.00, 7.108, 7.320],
    [0.01, 7.088, 7.306],
    [0.02, 7.027, 7.260],
    [0.03, 6.927, 7.184],
    [0.04, 6.792, 7.081],
    [0.05, 6.629, 6.953],
    [0.06, 6.441, 6.803],
    [0.07, 6.236, 6.634],
    [0.08, 6.017, 6.449],
    [0.09, 5.792, 6.253],
    [0.10, 5.564, 6.048],
    [0.11, 5.337, 5.838],
    [0.12, 5.113, 5.625],
    [0.13, 4.896, 5.411],
    [0.14, 4.686, 5.200],
    [0.15, 4.486, 4.992],
    [0.16, 4.295, 4.789],
    [0.17, 4.114, 4.593],
    [0.18, 3.942, 4.404],
    [0.19, 3.781, 4.222],
    [0.20, 3.629, 4.048],
    [0.22, 3.352, 3.724],
    [0.24, 3.108, 3.433],
    [0.25, 2.997, 3.299],
    [0.26, 2.892, 3.172],
    [0.28, 2.701, 2.940],
    [0.30, 2.531, 2.733],
    [0.32, 2.379, 2.548],
    [0.34, 2.242, 2.384],
    [0.35, 2.179, 2.308],
    [0.36, 2.119, 2.237],
    [0.38, 2.006, 2.105],
    [0.40, 1.903, 1.986],
    [0.42, 1.808, 1.878],
    [0.44, 1.720, 1.780],
    [0.45, 1.679, 1.734],
    [0.46, 1.639, 1.691],
    [0.48, 1.563, 1.608],
    [0.50, 1.492, 1.533],
    [0.55, 1.334, 1.367],
    [0.60, 1.197, 1.228],
    [0.65, 1.078, 1.108],
    [0.70, 0.973, 1.004],
    [0.80, 0.800, 0.832],
    [0.90, 0.665, 0.697],
    [1.00, 0.558, 0.589],
    [1.10, 0.473, 0.502],
    [1.20, 0.405, 0.431],
    [1.30, 0.350, 0.374],
    [1.40, 0.306, 0.327],
    [1.50, 0.270, 0.288],
    [1.60, 0.240, 0.256],
    [1.70, 0.215, 0.229],
    [1.80, 0.194, 0.206],
    [1.90, 0.175, 0.187],
    [2.00, 0.160, 0.170],
]) #From the International Tables for Crystallography, Volume C, 2006, p. 554.


def atomic_scattering_factors(h, k, l):
    s = np.sqrt(h**2 + k**2 + l**2) / (2 * a)
    idx = np.argmin(np.abs(scattering_table[:, 0] - s))
    f_Ga = scattering_table[idx, 1]
    f_As = scattering_table[idx, 2]

    return f_Ga, f_As


def fcc_allowed(h, k, l):
    parity = (abs(h) % 2, abs(k) % 2, abs(l) % 2)
    return parity[0] == parity[1] == parity[2]

@lru_cache(maxsize=None)
def U_hkl(h, k, l):
    if not fcc_allowed(h, k, l):
        return 0.0 + 0.0j

    f_Ga, f_As = atomic_scattering_factors(h, k, l)
    phase = np.exp(2j * np.pi * (h + k + l) / 4)
    F_hkl = 4 * (f_Ga + f_As * phase)
    V_cell = a**3
    U_g = F_hkl / (np.pi * V_cell)

    return U_g



## Zone-axis orientation and rotation matrices (tried to simulate Recipro orientation)


The orientation angles are calculated as

$$
\Theta
=
\operatorname{atan2}
\left(
\sqrt{u_n^2+v_n^2},
w_n
\right),
$$

and

$$
\Psi
=
\operatorname{atan2}(u_n,v_n).
$$

In the present calculation,

$$
\Phi = 0.
$$


In [5]:
def Rx(beta):
    return np.array([
        [1, 0,              0],
        [0, np.cos(beta), -np.sin(beta)],
        [0, np.sin(beta),  np.cos(beta)]
    ])


def Ry(beta):
    return np.array([
        [ np.cos(beta), 0, np.sin(beta)],
        [0,             1, 0],
        [-np.sin(beta), 0, np.cos(beta)]
    ])


def Rz(beta):
    return np.array([
        [np.cos(beta), -np.sin(beta), 0],
        [np.sin(beta),  np.cos(beta), 0],
        [0,             0,            1]
    ])



def zone_to_angles(u, v, w):
    direction = np.array([u, v, w], dtype=float)
    direction /= np.linalg.norm(direction)
    u_n, v_n, w_n = direction

    phi_deg = 0.0
    theta_deg = np.degrees(np.arctan2(np.hypot(u_n, v_n), w_n))
    psi_deg = np.degrees(np.arctan2(u_n, v_n))

    return phi_deg, theta_deg, psi_deg


phi_deg, theta_deg, psi_deg = zone_to_angles(u, v, w)

Phi   = np.deg2rad(phi_deg) # until now it's useless, though I thought this would be relevant for converged beam, but i solved it differently in the end. 
Theta = np.deg2rad(theta_deg)
Psi   = np.deg2rad(psi_deg) 

print("Corresponding Zoneanlge in Recipro would be following")
print(f"phi={phi_deg}°, theta={theta_deg}°, psi={psi_deg}°")

Corresponding Zoneanlge in Recipro would be following
phi=0.0°, theta=90.0°, psi=45.0°


<center>

## Excitation Error $s_g$
</center>

This still needs to be heavily changed in the end the excitation error $s_g$ is dependent on 
$$
    s_g\approx-\frac{2\mathbf{k}_0\cdot\mathbf{g}+|\mathbf{g|^2}}{2K}
$$

and dependent on the diffraction spot vector


\begin{equation}
    \mathbf{g}=\frac{1}{a}
    \begin{pmatrix}
        h\\
        k\\
        l\\
    \end{pmatrix}.
\end{equation}





and the wavevector 
\begin{equation}
    \mathbf{k}_0=K\cdot \mathbf{R}_z(-\Psi)\cdot\mathbf{R}_x(-\Theta)\cdot \mathbf{R}_z(-\Phi) \cdot \hat{\mathbf{e}}_\mathrm{z} 
    ,
\end{equation}



which depends on the Off **Zone angle** $\neq$ **Tilt angle** which getting into the equation via the Rotation Matrices

\begin{equation}
\begin{aligned}
\mathbf{R}_x(\beta)=
\begin{pmatrix}
1 & 0 & 0 \\
0 & \cos\beta & -\sin\beta \\
0 & \sin\beta & \cos\beta
\end{pmatrix}
\qquad
\mathbf{R}_z(\beta)=
\begin{pmatrix}
\cos\beta & -\sin\beta & 0 \\
\sin\beta & \cos\beta & 0\\
0 & 0 & 1 
\end{pmatrix}
\end{aligned}
\end{equation}

<!-- \begin{equation}
\mathbf{R}_y(\beta_y)=
\begin{pmatrix}
\cos\beta_y & 0 & \sin\beta_y \\
0 & 1 & 0 \\
-\sin\beta_y & 0 & \cos\beta_y
\end{pmatrix}
\end{equation} -->

With the Orientation of the beam to the crystal

\begin{equation}
    \hat{\mathbf{z}}_\mathrm{zone}=\frac{1}{\sqrt{h^2+k^2+l^2}}\cdot
    \begin{pmatrix}
        h\\
        k\\
        l\\
    \end{pmatrix}.
\end{equation}

To restrict the Bloch-wave calculation to the most relevant beams, each allowed reflection is assigned an importance value

$$
I_g
=
\frac{|U_g|}
{\sqrt{s_g^2+s_{\mathrm{floor}}^2}},
$$

where $U_g$ describes the strength of the crystal potential for the reflection. The small constant $s_{\mathrm{floor}}$ prevents a divergence for $s_g\rightarrow0$.

The reflections are ranked according to $I_g$. The strongest reflections are retained for the Bloch-wave calculation, while reflections with the same importance as the final selected beam are included as well. The direct beam $(000)$ is added separately.


In [6]:
# Incident electron wave vector
def k0(phi, theta, psi):
    ez = np.array([0.0, 0.0, 1.0])
    k_hat = Rz(-psi) @ Rx(-theta) @ Rz(-phi) @ ez
    k_vec = K * k_hat

    return k_vec

# Generate diffraction beams for this orientation. THIS IS DEFINITELY IMPROVABLE
def generate_g_list(
    theta_deg,
    psi_deg,
    phi_deg=0.0,
    n_beams=N_BEAMS,
    hkl_max=HKL_MAX,
    s_floor=1e-4):

    
    Phi   = np.deg2rad(phi_deg)
    Theta = np.deg2rad(theta_deg)
    Psi   = np.deg2rad(psi_deg)

    # Current incident beam vector
    k_vec = k0(Phi, Theta, Psi)

    candidates = []

    # Generate candidate GaAs reflections
    for h in range(-hkl_max, hkl_max + 1):
        for k in range(-hkl_max, hkl_max + 1):
            for l in range(-hkl_max, hkl_max + 1):

                # Skip direct beam, gets added later
                if (h, k, l) == (0, 0, 0):
                    continue

                # FCC reflection condition
                if not fcc_allowed(h, k, l):
                    continue


                # Reciprocal vector [Å^-1]
                g_vec = np.array([h, k, l], dtype=float) / a


                # Excitation error for this orientation
                s_g = -(2.0 * np.dot(k_vec, g_vec)+ np.dot(g_vec, g_vec)) / (2.0 * K)


                # Structure potential
                Ug = U_hkl(h, k, l)
                U_abs = abs(Ug)

                # Ignore effectively zero structure factors
                if U_abs < 1e-12:
                    continue

                #  IMPORTANCE
                importance = (U_abs/ np.sqrt(s_g**2 + s_floor**2))


                candidates.append({
                    "h": h,
                    "k": k,
                    "l": l,
                    "s_g": s_g,
                    "abs_s_g": abs(s_g),
                    "U_g": Ug,
                    "abs_U_g": U_abs,
                    "importance": importance
                })


    # Rank candidate beams
    beam_info = pd.DataFrame(candidates)

    beam_info = (
        beam_info
        .sort_values(
            "importance",
            ascending=False
        )
        .reset_index(drop=True)
    )


     # Importance value of the nominal last beam                                ############## Not a strict cutoff points after n_beams checks if following beams are in the same importance "magnitude" or "value" group and decice if it should take them as well under consideration
    cutoff = beam_info.iloc[n_beams - 2]["importance"]

    beam_info = beam_info[
        (beam_info["importance"] > cutoff)
        | np.isclose(
            beam_info["importance"],
            cutoff,
            rtol=1e-9,
            atol=1e-12
        )
    ].copy()


    # Construct g-list
    g_list = [(0, 0, 0)]
    g_list += [(int(row.h), int(row.k), int(row.l)) for row in beam_info.itertuples()]
    g_vectors = np.asarray(g_list, dtype=float) / a


    return g_list, g_vectors, beam_info

# Local basis for the orientation
def build_local_basis(theta_deg, psi_deg, phi_deg=0.0):
    
    g_list_local, g_vectors_local, beam_info_local = generate_g_list(theta_deg=theta_deg, psi_deg=psi_deg, phi_deg=phi_deg, n_beams=N_BEAMS, hkl_max=HKL_MAX)
    N_local = len(g_list_local)

    # |g|^2 for every local reciprocal vector
    g2_local = np.sum( g_vectors_local**2, axis=1)

    return (g_list_local, g_vectors_local, g2_local, beam_info_local)



# Test for the Orientation of the ZONE that was typed in 
g_list, g_vectors, g2, beam_info = build_local_basis(theta_deg=theta_deg, psi_deg=psi_deg, phi_deg=phi_deg)



print("Number of beams including 000:", len(g_list))
print(f"[{u}{v}{w}] g-vectors:")
print(g_list)

# You can check if the correct Diffractonspots are used or when it cuts of .head(20) or . tail(20)
display(beam_info[["h", "k", "l", "s_g", "abs_U_g", "importance"]].tail(20)) 



# Uptdating could make the code more efficent!
g_vectors = np.asarray(g_list, dtype=float) / a

s_g = {}

for g, g_vec in zip(g_list, g_vectors):
    s_g[g] = -(2*np.dot(k0(Phi, Theta, Psi), g_vec)
               + np.dot(g_vec, g_vec)) / (2*K)

Number of beams including 000: 21
[110] g-vectors:
[(0, 0, 0), (1, -1, -1), (-1, 1, -1), (1, -1, 1), (-1, 1, 1), (2, -2, 0), (-2, 2, 0), (1, -1, -3), (-1, 1, -3), (1, -1, 3), (-1, 1, 3), (0, 0, -4), (0, 0, 4), (3, -3, -1), (3, -3, 1), (-3, 3, -1), (-3, 3, 1), (2, -2, -4), (-2, 2, -4), (2, -2, 4), (-2, 2, 4)]


,h,k,l,s_g,abs_U_g,importance
0,1,-1,-1,-0.001178,0.047303,40.006183
1,-1,1,-1,-0.001178,0.047303,40.006183
2,1,-1,1,-0.001178,0.047303,40.006183
3,-1,1,1,-0.001178,0.047303,40.006183
4,2,-2,0,-0.003142,0.044375,14.116995
5,-2,2,0,-0.003142,0.044375,14.116995
6,1,-1,-3,-0.004320,0.026254,6.075735
7,-1,1,-3,-0.004320,0.026254,6.075735
8,1,-1,3,-0.004320,0.026254,6.075735
9,-1,1,3,-0.004320,0.026254,6.075735


## Bloch-wave eigenvalue problem

The propagation of the electron wave inside the crystal is described using a Bloch-wave expansion.  
For the selected set of reciprocal-lattice vectors, the Bloch coefficients are obtained by solving the eigenvalue problem

$$
\mathbf{A}\mathbf{C}^{(j)}
=
\gamma^{(j)}\mathbf{C}^{(j)}.
$$

Here, $\gamma^{(j)}$ is the eigenvalue of Bloch wave $j$, while $\mathbf{C}^{(j)}$ contains the corresponding Fourier coefficients of that Bloch wave.

The diagonal elements of $\mathbf{A}$ are given by the excitation errors $s_g$, while the off-diagonal elements describe coupling between different diffracted beams through the crystal potential:

$$
\begin{pmatrix}
s_{000}
& \frac{U_{-g_1}}{2K}
& \frac{U_{-g_2}}{2K}
& \frac{U_{-g_3}}{2K}
& \dots
& \frac{U_{-g_N}}{2K}
\\
\frac{U_{g_1}}{2K}
& s_{g_1}
& \frac{U_{g_1-g_2}}{2K}
& \frac{U_{g_1-g_3}}{2K}
& \dots
& \frac{U_{g_1-g_N}}{2K}
\\
\frac{U_{g_2}}{2K}
& \frac{U_{g_2-g_1}}{2K}
& s_{g_2}
& \frac{U_{g_2-g_3}}{2K}
& \dots
& \frac{U_{g_2-g_N}}{2K}
\\
\frac{U_{g_3}}{2K}
& \frac{U_{g_3-g_1}}{2K}
& \frac{U_{g_3-g_2}}{2K}
& s_{g_3}
& \dots
& \frac{U_{g_3-g_N}}{2K}
\\
\vdots
& \vdots
& \vdots
& \vdots
& \ddots
& \vdots
\\
\frac{U_{g_N}}{2K}
& \frac{U_{g_N-g_1}}{2K}
& \frac{U_{g_N-g_2}}{2K}
& \frac{U_{g_N-g_3}}{2K}
& \dots
& s_{g_N}
\end{pmatrix}
\begin{pmatrix}
C_{000}^{(j)} \\
C_{g_1}^{(j)} \\
C_{g_2}^{(j)} \\
C_{g_3}^{(j)} \\
\vdots \\
C_{g_N}^{(j)}
\end{pmatrix}
=
\gamma^{(j)}
\begin{pmatrix}
C_{000}^{(j)} \\
C_{g_1}^{(j)} \\
C_{g_2}^{(j)} \\
C_{g_3}^{(j)} \\
\vdots \\
C_{g_N}^{(j)}
\end{pmatrix}.
$$

### Excitation of the Bloch waves

The eigenvectors describe the possible Bloch waves inside the crystal. Their relative excitation amplitudes $\varepsilon^{(j)}$ are determined from the boundary condition at the entrance surface.

Before entering the crystal, only the direct beam $(000)$ is present. Therefore,

$$
\mathbf{\Psi}(0)=
\begin{pmatrix}
1\\
0\\
0\\
0\\
\vdots
\end{pmatrix}.
$$

The incident wave is expressed as a linear combination of the Bloch-wave eigenvectors:

$$
\mathbf{C}\mathbf{\varepsilon}
=
\mathbf{\Psi}(0).
$$

Hence,

$$
\mathbf{\varepsilon}
=
\mathbf{C}^{-1}\mathbf{\Psi}(0).
$$

In the numerical implementation, this system is solved directly rather than explicitly calculating the inverse of $\mathbf{C}$.

### Propagation through the crystal

Each Bloch wave accumulates a phase depending on its eigenvalue $\gamma^{(j)}$ and the propagation depth $z$.

The amplitude of reciprocal-lattice beam $g$ at depth $z$ is

$$
\psi_g(z)
=
\sum_j
C_g^{(j)}
\varepsilon^{(j)}
\exp\left(2\pi i\gamma^{(j)}z\right).
$$

The real-space electron wavefunction is then reconstructed by summing over all reciprocal-lattice beams:

$$
\mathbf{\Psi}(\mathbf{r},z)
=
\sum_g
\psi_g(z)
\exp\left(2\pi i\mathbf{g}\cdot\mathbf{r}\right).
$$

Finally, the local electron intensity is

$$
I(\mathbf{r},z)
=
\left|
\mathbf{\Psi}(\mathbf{r},z)
\right|^2.
$$

In [7]:
def solve_bloch(g_list, s_g, U_hkl, K):

    N = len(g_list)
    A = np.zeros((N, N), dtype=complex)

    for i, g in enumerate(g_list):
        for j, h in enumerate(g_list):
            if g == h:
                A[i, j] = s_g[g]
            else:
                dg = tuple(h[k] - g[k] for k in range(3))
                A[i, j] = U_hkl(*dg) / (2 * K)

    eigenvalues, eigenvectors = np.linalg.eigh(A)

    bloch_psi_0 = np.zeros(N, dtype=complex)
    bloch_psi_0[g_list.index((0, 0, 0))] = 1

    epsilon = np.linalg.solve(eigenvectors, bloch_psi_0)

    return eigenvalues, eigenvectors, epsilon
    
gamma, C, epsilon = solve_bloch(g_list, s_g, U_hkl, K)


def calc_bloch_psi_g(z):
    """
    Diffracted-beam amplitudes bloch_psi_g at depth z.
    """
    phase = np.exp(2j * np.pi * gamma * z)
    bloch_psi_g = C @ (epsilon * phase)
    return bloch_psi_g

    # gamma = eigenvelues
    # C = eigenvectors

def calc_bloch_Psi(r, z):
    bloch_psi_g = calc_bloch_psi_g(z)
    spatial_phase = np.exp(2j * np.pi * (g_vectors @ r))
    return np.sum(bloch_psi_g * spatial_phase)


## Thickness-averaged intensity at the Ga and As sites

For each beam orientation, the relevant diffraction beams are selected again and the corresponding Bloch-wave eigenvalue problem is solved.

For an atomic position $\mathbf{r}_c$, the contribution of Bloch wave $j$ is

$$
A_c^{(j)}
=
\varepsilon^{(j)}
\sum_g
C_g^{(j)}
\exp\left(2\pi i\,\mathbf{g}\cdot\mathbf{r}_c\right).
$$

The electron intensity at this atomic position varies with depth because the different Bloch waves accumulate different phases.  
Rather than evaluating the wavefunction at many individual depths, the intensity is averaged analytically over the specimen thickness $t$.

For two Bloch waves $i$ and $j$, define

$$
\Delta\gamma_{ij}
=
\gamma^{(i)}-\gamma^{(j)}.
$$

The thickness average of their interference term is

$$
T_{ij}
=
\frac{1}{t}
\int_0^t
\exp\left(2\pi i\,\Delta\gamma_{ij}z\right)\,dz.
$$

This gives

$$
T_{ij}
=
\exp\left(i\pi\Delta\gamma_{ij}t\right)
\operatorname{sinc}\left(\Delta\gamma_{ij}t\right),
$$

where

$$
\operatorname{sinc}(x)
=
\frac{\sin(\pi x)}{\pi x}.
$$

The thickness-averaged electron intensity at atomic column $c$ is therefore

$$
J_c
=
\sum_{i,j}
A_c^{(i)}
T_{ij}
A_c^{(j)*}.
$$




In [8]:
def calc_intensity_map(z, x_range=(-5, 5), y_range=(-5, 5), N=200):

    # Current zone axis from u,v,w
    n_zone = np.array([u, v, w], dtype=float)
    n_zone /= np.linalg.norm(n_zone)

    # Choose a reference not parallel to the zone
    ref = np.array([0.0, 0.0, 1.0])

    if abs(np.dot(n_zone, ref)) > 0.99:
        ref = np.array([0.0, 1.0, 0.0])

    # Plane perpendicular to zone axis
    e_x = np.cross(ref, n_zone)
    e_x /= np.linalg.norm(e_x)

    e_y = np.cross(n_zone, e_x)
    e_y /= np.linalg.norm(e_y)

    # Real-space grid
    x = np.linspace(*x_range, N)
    y = np.linspace(*y_range, N)
    X, Y = np.meshgrid(x, y)

    # Bloch-wave amplitudes at depth z
    bloch_psi_g = calc_bloch_psi_g(z)

    # Real-space positions
    R = (
        X[..., None] * e_x
        + Y[..., None] * e_y
        + z * n_zone
    )

    g_dot_r = np.einsum("gc,xyc->xyg", g_vectors, R)

    phase = np.exp(2j * np.pi * g_dot_r)

    bloch_Psi = np.sum(
        bloch_psi_g[None, None, :] * phase,
        axis=2
    )

    I = np.abs(bloch_Psi)**2

    return X, Y, I

X, Y, I = calc_intensity_map(thickness_A)

I_rel = 100 * (I - np.mean(I)) / np.mean(I)

fig = go.Figure(
    data=[
        go.Surface(
            x=X,
            y=Y,
            z=I_rel,
            colorscale="Viridis"
        )
    ]
)

fig.update_layout(
    title=f"Zone axis [{u}{v}{w}] with Psi {psi_deg:.2f}°, Theta {theta_deg:.2f}°",
    scene=dict(
        xaxis_title="x direction (Å)",
        yaxis_title="y direction (Å)",
        zaxis_title="Intensity deviation (%)"
    ),
    width=1000,
    height=800
)
pio.renderers.default = "iframe"
fig.show()

## Thickness-averaged intensity at the atomic sites

The zinc-blende unit cell contains four Ga and four As positions. For each atomic position
$\mathbf{R}_c$, the Bloch-wave amplitudes are combined with the corresponding spatial phase

$$
\exp\left(2\pi i\,\mathbf{g}\cdot\mathbf{R}_c\right).
$$

For Bloch wave $j$, the amplitude at atomic site $c$ is therefore

$$
A_c^{(j)}
=
\varepsilon^{(j)}
\sum_g
C_g^{(j)}
\exp\left(2\pi i\,\mathbf{g}\cdot\mathbf{R}_c\right).
$$

The electron intensity is averaged over the specimen thickness $t$:

$$
J_c
=
\frac{1}{t}
\int_0^t
\left|\Psi(\mathbf{R}_c,z)\right|^2\,dz.
$$

With

$$
\Delta\gamma_{jl}
=
\gamma_j-\gamma_l,
$$

the analytical thickness-averaging term becomes

$$
T_{jl}
=
\exp\left[i\pi(\gamma_j-\gamma_l)t\right]
\operatorname{sinc}\left[(\gamma_j-\gamma_l)t\right],
$$

where NumPy uses

$$
\operatorname{sinc}(x)
=
\frac{\sin(\pi x)}{\pi x}.
$$

The thickness-averaged intensity at atomic site $c$ is then

$$
J_c
=
\sum_{j,l}
A_c^{(j)}
A_c^{(l)*}
T_{jl}.
$$

The four Ga and As site intensities are averaged separately,

$$
J_{\mathrm{Ga}}
=
\frac{1}{4}\sum_{c\in\mathrm{Ga}}J_c,
\qquad
J_{\mathrm{As}}
=
\frac{1}{4}\sum_{c\in\mathrm{As}}J_c.
$$

The channeling factor plotted in the rocking map is defined as

$$
f_{\mathrm{channeling}}
=
\frac{J_{\mathrm{As}}}{J_{\mathrm{Ga}}}.
$$

In [9]:
Ga_positions = a * np.array([
    [0.0, 0.0, 0.0],
    [0.0, 0.5, 0.5],
    [0.5, 0.0, 0.5],
    [0.5, 0.5, 0.0]
])

As_positions = a * np.array([
    [0.25, 0.25, 0.25],
    [0.25, 0.75, 0.75],
    [0.75, 0.25, 0.75],
    [0.75, 0.75, 0.25]
])


def project_to_column(R, n_zone):
    
    return R - np.dot(R, n_zone) * n_zone


R_Ga_columns = np.array([project_to_column(R, n_zone) for R in Ga_positions])
R_As_columns = np.array([project_to_column(R, n_zone) for R in As_positions])


def calc_f_channeling_parallel(theta_deg, psi_deg, phi_deg=0.0):

    # Generate diffraction beams for THIS orientation
    (g_list_local, g_vectors_local, g2_local, beam_info_local) = build_local_basis(theta_deg=theta_deg, psi_deg=psi_deg, phi_deg=phi_deg)

    N_local = len(g_list_local)



    # Incident beam direction
    Phi   = np.deg2rad(phi_deg)
    Theta = np.deg2rad(theta_deg)
    Psi   = np.deg2rad(psi_deg)

    k_vec_local = k0(Phi,Theta,Psi)

    # Excitation errors for LOCAL g-vectors
    s_local = -(2.0 * (g_vectors_local @ k_vec_local) + g2_local) / (2.0 * K)


    # Build LOCAL Bloch matrix
    A_local = np.zeros((N_local, N_local), dtype=complex)

    # diagonal = excitation errors
    np.fill_diagonal(A_local, s_local)


    # off-diagonal coupling
    for i, g in enumerate(g_list_local):
        for j, h in enumerate(g_list_local):

            if i != j:
                h_minus_g = (
                    h[0] - g[0],
                    h[1] - g[1],
                    h[2] - g[2]
                )

                A_local[i, j] = (U_hkl(*h_minus_g)/ (2.0 * K))


    # (000) is always first in generate_g_list()
    psi_0_local = np.zeros(N_local, dtype=complex)
    psi_0_local[0] = 1.0


    # Solve Bloch-wave eigenvalue problem
    gamma_local, C_local = np.linalg.eigh(A_local)
    epsilon_local = (C_local.conj().T @ psi_0_local)

    # Thickness factor
    delta_gamma = (gamma_local[:, None] - gamma_local[None, :])
    thickness_factor = (np.exp(1j * np.pi * delta_gamma * thickness_A) *np.sinc(delta_gamma * thickness_A))

    # Atomic-site phase factors
    phase_Ga_local = np.exp(2j * np.pi * (Ga_positions @ g_vectors_local.T))
    phase_As_local = np.exp(2j * np.pi * (As_positions @ g_vectors_local.T))


    # Ga column intensities
    A_Ga = (phase_Ga_local @ C_local) * epsilon_local[None, :]
    J_Ga_columns = np.einsum("ci,ij,cj->c", A_Ga, thickness_factor, np.conj(A_Ga))
    J_Ga_local = np.mean(np.real(J_Ga_columns))

    # As column intensities
    A_As = (phase_As_local @ C_local) * epsilon_local[None, :]
    J_As_columns = np.einsum("ci,ij,cj->c", A_As, thickness_factor, np.conj(A_As))
    J_As_local = np.mean(np.real(J_As_columns))



    # Channeling factor
    f_local = (J_As_local/ J_Ga_local)

    return (J_Ga_local, J_As_local, f_local)


J_Ga, J_As, f_channeling = calc_f_channeling_parallel(
    theta_deg=theta_deg,
    psi_deg=psi_deg,
    phi_deg=phi_deg
)
print(f"Exactly on Zone of [{u}{v}{w}]:")
print("J_Ga =", J_Ga)
print("J_As =", J_As)
print("f_channeling =", f_channeling)


Exactly on Zone of [110]:
J_Ga = 3.6570631062048204
J_As = 4.147982337905143
f_channeling = 1.1342386547465904


# Rocking Map Calculation

In [ ]:
scan_range1=scan_range1 + 00.1 # this is just because otherwise the last step will be skipped
scan_range2=scan_range2 + 00.1 # this is just because otherwise the last step will be skipped
dtheta = np.arange(-scan_range1, scan_range1, stepsize)
dpsi   = np.arange(-scan_range2, scan_range2, stepsize)
n_total = len(dpsi) * len(dtheta)

print(f"Start: {n_total:,} pixels")


def calculate_row(dp):

    J_Ga_row = np.zeros(len(dtheta))
    J_As_row = np.zeros(len(dtheta))

    for j, dt in enumerate(dtheta):

        theta = theta_deg + dt
        psi   = psi_deg + dp

        J_Ga, J_As, _ = calc_f_channeling_parallel(
            theta_deg=theta,
            psi_deg=psi
        )

        J_Ga_row[j] = J_Ga
        J_As_row[j] = J_As

    return J_Ga_row, J_As_row

    
start_time = time.time()


# Map Calculation with Progress bars
Ga_rows = []
As_rows = []

pbar = tqdm(
    total=n_total,
    desc="Channeling map",
    unit="pixel"
)

for dp in dpsi:

    J_Ga_row, J_As_row = calculate_row(dp)

    Ga_rows.append(J_Ga_row)
    As_rows.append(J_As_row)

    pbar.update(len(dtheta))

pbar.close()


# 2D GLOBAL ORIENTATION MAP
# Convert calculated rows to maps
J_Ga_map = np.array(Ga_rows)
J_As_map = np.array(As_rows)

f_map = J_As_map / J_Ga_map

print(f"Finished in {(time.time() - start_time) / 3600:.2f} hours")


# Global angles
theta_global = theta_deg + dtheta
psi_global   = psi_deg + dpsi

THETA, PSI = np.meshgrid(theta_global, psi_global)

angle_data = np.stack((THETA, PSI), axis=-1)


# Interactive 2D channeling map
fig = go.Figure(
    go.Heatmap(
        x=theta_global,
        y=psi_global,
        z=f_map,

        customdata=angle_data,

        colorscale="Viridis",

        colorbar=dict(
            title="J_As / J_Ga"
        ),

        hovertemplate=(
            "Theta = %{customdata[0]:.3f}°<br>"
            "Psi = %{customdata[1]:.3f}°<br>"
            "f = %{z:.6f}"
            "<extra></extra>"
        )
    )
)


fig.update_layout(
    title=f"f_channeling around [{u}{v}{w}] zone axis",
    width=900,
    height=800
)

fig.update_xaxes(
    title="Theta (deg)"
)

fig.update_yaxes(
    title="Psi (deg)",
    scaleanchor="x",
    scaleratio=1
)


print("f min :", np.nanmin(f_map))
print("f max :", np.nanmax(f_map))
print("range :", np.nanmax(f_map) - np.nanmin(f_map))
print("mean  :", np.nanmean(f_map))


pio.renderers.default = "iframe"
fig.show()



Start: 1,004,004 pixels


Channeling map:   1%|          | 9018/1004004 [01:45<3:13:59, 85.48pixel/s]

In [ ]:
# Sphere Illustration
DTHETA, DPSI = np.meshgrid(dtheta, dpsi)
THETA_deg = theta_deg + DTHETA
PSI_deg = psi_deg + DPSI
THETA = np.deg2rad(THETA_deg)
PSI = np.deg2rad(PSI_deg)

f_min = np.min(f_map)
f_max = np.max(f_map)
if f_max > f_min:
    f_norm = (f_map - f_min) / (f_max - f_min)
else:
    f_norm = np.zeros_like(f_map)

relief = 0.25
R = 1.0 + relief * f_norm

# [100] -> Theta=90°, Psi=90°; [010] -> Theta=90°, Psi=0°; [001] -> Theta=0°
X_sphere = R * np.sin(THETA) * np.sin(PSI)
Y_sphere = R * np.sin(THETA) * np.cos(PSI)
Z_sphere = R * np.cos(THETA)

# Reference sphere
theta_ref = np.linspace(0, np.pi, 100)
psi_ref = np.linspace(0, 2 * np.pi, 200)
TH_ref, PS_ref = np.meshgrid(theta_ref, psi_ref)
X_ref = np.sin(TH_ref) * np.sin(PS_ref)
Y_ref = np.sin(TH_ref) * np.cos(PS_ref)
Z_ref = np.cos(TH_ref)

fig_globe = go.Figure()
fig_globe.add_trace(go.Surface(
    x=X_ref,
    y=Y_ref,
    z=Z_ref,
    surfacecolor=np.zeros_like(X_ref),
    colorscale=[[0, "lightgray"], [1, "lightgray"]],
    showscale=False,
    opacity=0.10,
    hoverinfo="skip",
    name="Orientation sphere"
))

fig_globe.add_trace(go.Surface(
    x=X_sphere,
    y=Y_sphere,
    z=Z_sphere,
    surfacecolor=f_map,
    colorscale="Viridis",
    customdata=np.stack((THETA_deg, PSI_deg, f_map), axis=-1),
    hovertemplate=(
        "Theta = %{customdata[0]:.2f}°<br>"
        "Psi = %{customdata[1]:.2f}°<br>"
        "f = %{customdata[2]:.4f}<extra></extra>"
    ),
    colorbar=dict(title="J_As / J_Ga"),
    name="f_channeling"
))

# Low-index zone axes with h,k,l in {-1,0,1}, excluding [000]
zones = {
    f"[{h}{k}{l}]": np.array([h, k, l], dtype=float)
    for h in (-1, 0, 1)
    for k in (-1, 0, 1)
    for l in (-1, 0, 1)
    if (h, k, l) != (0, 0, 0)
}

zone_radius = 1.0 + relief + 0.08
zone_names = []
zone_xyz = []

for name, direction in zones.items():
    direction = direction / np.linalg.norm(direction)
    zone_names.append(name)
    zone_xyz.append(zone_radius * direction)

zone_xyz = np.asarray(zone_xyz)
fig_globe.add_trace(go.Scatter3d(
    x=zone_xyz[:, 0],
    y=zone_xyz[:, 1],
    z=zone_xyz[:, 2],
    mode="markers+text",
    text=zone_names,
    textposition="top center",
    marker=dict(size=4, symbol="circle"),
    hovertemplate="%{text}<extra></extra>",
    name="Zone axes"
))

for direction in zones.values():
    direction = direction / np.linalg.norm(direction)
    fig_globe.add_trace(go.Scatter3d(
        x=[direction[0], zone_radius * direction[0]],
        y=[direction[1], zone_radius * direction[1]],
        z=[direction[2], zone_radius * direction[2]],
        mode="lines",
        line=dict(width=2),
        showlegend=False,
        hoverinfo="skip"
    ))

axis_limit = 1.0 + relief + 0.25
fig_globe.update_layout(
    title=f"Spherical f_channeling map around [{u}{v}{w}]",
    scene=dict(
        xaxis_title="[100]",
        yaxis_title="[010]",
        zaxis_title="[001]",
        aspectmode="cube",
        xaxis=dict(range=[-axis_limit, axis_limit]),
        yaxis=dict(range=[-axis_limit, axis_limit]),
        zaxis=dict(range=[-axis_limit, axis_limit]),
        camera=dict(projection=dict(type="orthographic"))
    ),
    width=1000,
    height=1000,
    showlegend=True
)
fig_globe.show()

In [ ]:
save_dir = Path("/cluster/work/gregorg/master_thesis/data/f_channeling")
save_dir.mkdir(parents=True, exist_ok=True)


filename = (
    pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M-%S")
    + f"_f_channeling_[{u}{v}{w}]_dataset_{scan_range1:.0f}x{scan_range2:.0f}_in_{stepsize}_{special_name_for_file}"
    + ".npz"
)


np.savez(
    save_dir / filename,

    dtheta=dtheta,
    dpsi=dpsi,

    J_Ga_map=J_Ga_map,
    J_As_map=J_As_map,
    f_map=f_map,

    theta_deg=theta_deg,
    psi_deg=psi_deg,
    thickness_A=thickness_A
)

print("Dataset saved to:", save_dir / filename)